# Surveillance de Modeles de ML avec Prometheus et Grafana

## Contexte et Objectifs

Ce notebook est un guide pratique pour mettre en place un systeme de surveillance (monitoring) pour un modele de machine learning deploye en tant qu'API. La surveillance est une etape cruciale du MLOps qui permet de s'assurer que le modele fonctionne comme prevu en production, de detecter les derives (data drift, concept drift) et de suivre les performances.

Nous utiliserons une pile d'outils open-source tres populaire :
- **Flask :** Pour creer une API REST simple qui expose notre modele.
- **Prometheus :** Pour collecter les metriques de performance de notre API.
- **Grafana :** Pour visualiser ces metriques dans des tableaux de bord interactifs.

### Flux de Travail Aborde :

1.  **Creation d'une API Flask :** Nous developperons une petite application Flask qui charge un modele scikit-learn pre-entraine (par exemple, un classificateur sur le jeu de donnees Iris) et expose un point de terminaison `/predict`.
2.  **Instrumentation de l'API :** A l'aide de la bibliotheque `prometheus-flask-exporter`, nous ajouterons des metriques personnalisees a notre API pour suivre :
    *   La latence des requetes.
    *   Le nombre de predictions effectuees.
    *   La distribution des predictions (par classe).
    *   Un exemple de surveillance de la distribution d'une caracteristique d'entree pour detecter une derive de donnees.
3.  **Configuration de Prometheus :** Nous fournirons un fichier de configuration `prometheus.yml` pour que Prometheus puisse "gratter" (scraper) les metriques exposees par notre API Flask.
4.  **Lancement avec Docker Compose :** Pour simplifier le deploiement, nous utiliserons Docker Compose pour lancer l'API Flask, Prometheus et Grafana en une seule commande.
5.  **Creation d'un Tableau de Bord Grafana :** Nous expliquerons comment se connecter a Grafana, ajouter Prometheus comme source de donnees et creer des visualisations pour nos metriques.

_Derniere mise a jour : 2026-02-16_

## Etape 1: Creation de l'Application Flask et Instrumentation

Creez un fichier nomme `app.py`. Ce script definit notre API, charge un modele Iris et configure les metriques Prometheus.

In [1]:
# N'executez pas cette cellule directement dans le notebook.
# Enregistrez ce code dans un fichier nomme `app.py`

from flask import Flask, request, jsonify
from prometheus_flask_exporter import PrometheusMetrics
from prometheus_client import Counter, Histogram
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
import numpy as np

# --- 1. Entrainer et Sauvegarder un Modele Simple ---
X, y = load_iris(return_X_y=True)
model = LogisticRegression(max_iter=200)
model.fit(X, y)
iris_target_names = load_iris().target_names

# --- 2. Creer l'application Flask ---
app = Flask(__name__)

# --- 3. Instrumenter avec Prometheus ---
metrics = PrometheusMetrics(app)
# Metriques standard (requetes, latence, etc.) sont exposees automatiquement

# Metriques personnalisees
PREDICTIONS_BY_CLASS = Counter('predictions_by_class', 'Number of predictions by class', ['class'])
FEATURE_DISTRIBUTION = Histogram('feature_distribution', 'Distribution of a feature', ['feature_name'])

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    features = np.array(data['features']).reshape(1, -1)
    
    # Enregistrer la distribution de la premiere caracteristique
    FEATURE_DISTRIBUTION.labels(feature_name='sepal_length').observe(features[0][0])
    
    # Faire la prediction
    prediction_id = model.predict(features)[0]
    prediction_name = iris_target_names[prediction_id]
    
    # Incrementer le compteur pour la classe predite
    PREDICTIONS_BY_CLASS.labels(class_name=prediction_name).inc()
    
    return jsonify({'prediction': prediction_name})

@app.route('/')
def index():
    return "API de prediction du modele Iris. Utilisez le point de terminaison /predict."

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

✅ Output snapshot saved (sanitized): execution artifacts prepared for GitHub rendering.\n

## Etape 2: Configuration de Prometheus

Creez un fichier nomme `prometheus.yml`. Ce fichier indique a Prometheus ou trouver les metriques de notre application Flask.

In [2]:
# N'executez pas cette cellule directement dans le notebook.
# Enregistrez ce code dans un fichier nomme `prometheus.yml`

global:
  scrape_interval: 15s # Par defaut, scrape toutes les 15 secondes.

scrape_configs:
  - job_name: 'flask-app'
    # L'adresse de l'API Flask. Puisque nous utiliserons Docker Compose,
    # nous pouvons utiliser le nom du service 'flask_app' comme nom d'hote.
    static_configs:
      - targets: ['flask_app:5000']

✅ Output snapshot saved (sanitized): execution artifacts prepared for GitHub rendering.\n

## Etape 3: Orchestration avec Docker Compose

Creez un fichier nomme `docker-compose.yml` pour definir et lier nos trois services : l'API, Prometheus, et Grafana.

In [3]:
# N'executez pas cette cellule directement dans le notebook.
# Enregistrez ce code dans un fichier nomme `docker-compose.yml`

version: '3.7'

services:
  flask_app:
    build: . # On suppose que votre Dockerfile est dans le meme repertoire
    ports:
      - "5000:5000"

  prometheus:
    image: prom/prometheus:v2.26.0
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml
    ports:
      - "9090:9090"
    depends_on:
      - flask_app

  grafana:
    image: grafana/grafana:7.5.7
    ports:
      - "3000:3000"
    depends_on:
      - prometheus

✅ Output snapshot saved (sanitized): execution artifacts prepared for GitHub rendering.\n

## Etape 4: Creation du Dockerfile

Enfin, creez un fichier `Dockerfile` pour construire l'image de notre application Flask.

In [4]:
# N'executez pas cette cellule directement dans le notebook.
# Enregistrez ce code dans un fichier nomme `Dockerfile`

FROM python:3.8-slim

WORKDIR /app

COPY app.py .

# Installez les dependances
RUN pip install flask prometheus-flask-exporter scikit-learn numpy

# Exposez le port et lancez l'application
EXPOSE 5000
CMD ["python", "app.py"]

✅ Output snapshot saved (sanitized): execution artifacts prepared for GitHub rendering.\n

## Etape 5: Lancement et Visualisation

1.  **Lancez tout :** Ouvrez un terminal dans le repertoire ou vous avez cree ces quatre fichiers et executez :
    ```bash
    docker-compose up -d
    ```
2.  **Generez du trafic :** Envoyez quelques requetes a votre API.
    ```bash
    curl -X POST -H "Content-Type: application/json" -d '{"features": [5.1, 3.5, 1.4, 0.2]}' http://localhost:5000/predict
    ```
3.  **Explorez Prometheus :** Ouvrez votre navigateur et allez a `http://localhost:9090`. Vous pouvez interroger des metriques comme `flask_http_requests_total`.
4.  **Configurez Grafana :**
    *   Allez a `http://localhost:3000` (login: admin, password: admin).
    *   Allez dans `Configuration > Data Sources > Add data source`.
    *   Choisissez Prometheus.
    *   Pour l'URL, entrez `http://prometheus:9090`.
    *   Cliquez sur `Save & Test`.
    *   Creez un nouveau tableau de bord et ajoutez des panneaux en utilisant des requetes PromQL (par exemple, `rate(predictions_by_class[5m])`).